In [11]:
from datasets import load_dataset

ds = load_dataset("arbml/Arabic_Sentiment_Twitter_Corpus")

train_data = ds["train"]
test_data  = ds["test"]

Repo card metadata block was not found. Setting CardData to empty.


In [12]:
import re
from arabic_sentiment.preprocessing import ArabicPreprocessor
from arabic_sentiment.language_model import NgramLanguageModel

In [13]:
p = ArabicPreprocessor()

train_data = train_data.map(lambda x: {"clean_tweet": p.preprocess(x["tweet"])})

In [14]:
model_1 = NgramLanguageModel(2)
model_2 = NgramLanguageModel(2)
model_3 = NgramLanguageModel(3)
model_4 = NgramLanguageModel(3)

raw_train = [x.split() for x in train_data["tweet"]]
clean_train = train_data["clean_tweet"]

model_1.train(raw_train)
model_2.train(clean_train)
model_3.train(raw_train)
model_4.train(clean_train)


test_data = test_data.map(lambda x: {"clean_tweet": p.preprocess(x["tweet"])})
raw_test = [x.split() for x in test_data["tweet"]]
clean_test = test_data["clean_tweet"]

p_raw1 = model_1.perplexity(raw_test)
p_clean1 = model_2.perplexity(clean_test)
p_raw2 = model_3.perplexity(raw_test)
p_clean2 = model_4.perplexity(clean_test)

print("\n" + "="*45)
print(f"{'Model':<15} | {'Raw PPL':<12} | {'Preprocessed':<12}")
print("-" * 45)
print(f"{'Bigram':<15} | {p_raw1:<12.2f} | {p_clean1:<12.2f}")
print(f"{'Trigram':<15} | {p_raw2:<12.2f} | {p_clean2:<12.2f}")
print("="*45 + "\n")


Model           | Raw PPL      | Preprocessed
---------------------------------------------
Bigram          | 12344.66     | 9263.26     
Trigram         | 18800.94     | 12713.28    



In [16]:
models = {
    "Bigram (Raw)": model_1,
    "Bigram (Clean)": model_2,
    "Trigram (Raw)": model_3,
    "Trigram (Clean)": model_4
}

for name, model in models.items():
    print(f"--- Generated Samples from [{name}] ---")
    for i in range(3):
        sample = model.generate()
        print(f"{i+1}. {sample}")
    print("-" * 30)

--- Generated Samples from [Bigram (Raw)] ---
1. اضرب يا نور الرضوان إلى الطيب عزة 🌼💚🌼
2. بمناسبة فوز يعني لو معنا ذالليلة بمناسبة فوز الهلال تردد نشيد #الاتحاد #الشاهينو
3. اعتني بمن يتابعنا عبر #زواج_عبدالاله_الاركاني #همثون77
------------------------------
--- Generated Samples from [Bigram (Clean)] ---
1. امسينا و ابيها تتكلم ولكن اين السبيل الى غد السبت الا بالله يعنى شلته خلاص بسويلك
2. س الدهر كله عاجله واجله
3. واني اتنفس انا وكهربا طاقه وكهربا طاقه اعدي بيها نقص وقسوه الدنيا
------------------------------
--- Generated Samples from [Trigram (Raw)] ---
1. الحمد لله💙 ألف مبروك يازعماء عقبال الجاي إن شاء الله 🙏🏼
2. ياحظك وأنا أشهد 😭
3. نادره بالقصر والوجه الطفولي جعل يفداك العتاب و الك الرضا مني و أنا طفل فقطعت رجله بخيط 😡 ف…
------------------------------
--- Generated Samples from [Trigram (Clean)] ---
1. الجامعه اللي بالرياض تفتح النفس يارب لك الحمد والحمدلله بفضل الله ثم دعمكم تجاوزنا المباراه
2. اجوا الصباح والورد والقهوه
3. ماحددوا منهم الحكام اللي يبونهم يحكمون لهم بالمره
